# e5 extended-band scoring (D-029) — reuse the fine-tuned multilingual-e5-small (MIT) saved in My Drive/ber_ce/out/model
Needs: folder `ber_e5_ext` (3 score parquet files) uploaded to My Drive. Runtime → Change runtime type → **T4 GPU**. Runtime → Run all.
Writes `My Drive/ber_e5_ext/out/ce_{tune,holdout,test}.parquet` (~15 min).

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, glob, time, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
direct = '/content/drive/MyDrive/ber_ce/out/model'  # also works through a 'Shared with me' shortcut named ber_ce
cands = [direct + '/model.safetensors'] if os.path.exists(direct + '/model.safetensors') else \
        glob.glob('/content/drive/MyDrive/**/model/model.safetensors', recursive=True)
bund = glob.glob('/content/drive/MyDrive/**/ber_e5_ext/score_tune.parquet', recursive=True)
print('models found:', cands); print('bundle found:', bund)
print('My Drive top level:', os.listdir('/content/drive/MyDrive'))
MODEL = os.path.dirname(cands[0])
BUNDLE = os.path.dirname(bund[0])
OUT = BUNDLE + '/out'; os.makedirs(OUT, exist_ok=True)
assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> T4 GPU'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL).cuda().eval()
print('using', MODEL, BUNDLE)

In [ ]:
MAXLEN = 64  # same as training / first scoring
def evaluate(df):
    out = np.zeros(len(df), dtype=np.float32)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        for s in range(0, len(df), 1024):
            j = np.arange(s, min(s + 1024, len(df)))
            enc = tok(df.ta.values[j].tolist(), df.tb.values[j].tolist(), truncation=True, max_length=MAXLEN,
                      padding=True, return_tensors='pt')
            out[j] = model(**{k: v.cuda() for k, v in enc.items()}).logits.squeeze(-1).float().cpu().numpy()
    return out
for g in ('tune', 'holdout', 'test'):
    df = pd.read_parquet(f'{BUNDLE}/score_{g}.parquet')
    order = np.argsort((df.ta.str.len() + df.tb.str.len()).values, kind='stable')
    df = df.iloc[order].reset_index(drop=True)
    t0 = time.time(); ce = evaluate(df)
    pd.DataFrame({'lid': df.lid, 'rid': df.rid, 'ce': ce}).to_parquet(f'{OUT}/ce_{g}.parquet', index=False)
    print(g, len(df), f'{(time.time()-t0)/60:.1f} min', flush=True)
print('DONE — download My Drive/ber_e5_ext/out/ce_tune.parquet, ce_holdout.parquet, ce_test.parquet')